In [32]:
import os 
from dotenv import load_dotenv

from langchain_community.document_loaders import TextLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.embeddings import JinaEmbeddings
from langchain_community.vectorstores import FAISS
from langchain_groq import ChatGroq

In [9]:
load_dotenv()

True

In [10]:
groq_key = os.getenv("GROQ_API_KEY")
jina_key = os.getenv("JINA_API_KEY")

In [11]:
DATA_FILE_PATH = os.path.join("data", "hr_policy.txt")

In [12]:
DATA_FILE_PATH

'data\\hr_policy.txt'

In [13]:
loader = TextLoader(DATA_FILE_PATH, encoding="utf-8")

documents = loader.load()


print("DATA LOADDED")
print("="*40)
print(documents)

DATA LOADDED
[Document(metadata={'source': 'data\\hr_policy.txt'}, page_content='COMPANY HR POLICY HANDBOOK\nAcme Corp - Employee Handbook (Demo Document)\n\n1. LEAVE POLICY\nAll full-time employees are entitled to 20 days of paid annual leave per calendar year.\nLeave requests must be submitted through the HR portal at least 5 working days in advance.\nUnused annual leave can be carried forward to the next year, up to a maximum of 5 days.\nSick leave is separate from annual leave, and employees get 10 paid sick days per year.\nA medical certificate is required for sick leave longer than 2 consecutive days.\n\n2. WORK FROM HOME POLICY\nEmployees may work from home up to 2 days per week, subject to manager approval.\nFully remote work arrangements require written approval from the department head.\nEmployees working from home must be reachable during core hours: 10 AM to 4 PM.\n\n3. PROBATION PERIOD\nAll new employees undergo a probation period of 3 months from their date of joining.\nD

In [14]:
len(documents)

1

In [16]:
print(documents[0].page_content)

COMPANY HR POLICY HANDBOOK
Acme Corp - Employee Handbook (Demo Document)

1. LEAVE POLICY
All full-time employees are entitled to 20 days of paid annual leave per calendar year.
Leave requests must be submitted through the HR portal at least 5 working days in advance.
Unused annual leave can be carried forward to the next year, up to a maximum of 5 days.
Sick leave is separate from annual leave, and employees get 10 paid sick days per year.
A medical certificate is required for sick leave longer than 2 consecutive days.

2. WORK FROM HOME POLICY
Employees may work from home up to 2 days per week, subject to manager approval.
Fully remote work arrangements require written approval from the department head.
Employees working from home must be reachable during core hours: 10 AM to 4 PM.

3. PROBATION PERIOD
All new employees undergo a probation period of 3 months from their date of joining.
During probation, employees are not eligible for paid leave, but may take unpaid leave
in case of e

In [17]:
documents[0].metadata

{'source': 'data\\hr_policy.txt'}

In [19]:
text_spliter = RecursiveCharacterTextSplitter(
    chunk_size = 500,
    chunk_overlap = 50
)

chunks = text_spliter.split_documents(documents=documents)

In [20]:
print(chunks)

[Document(metadata={'source': 'data\\hr_policy.txt'}, page_content='COMPANY HR POLICY HANDBOOK\nAcme Corp - Employee Handbook (Demo Document)'), Document(metadata={'source': 'data\\hr_policy.txt'}, page_content='1. LEAVE POLICY\nAll full-time employees are entitled to 20 days of paid annual leave per calendar year.\nLeave requests must be submitted through the HR portal at least 5 working days in advance.\nUnused annual leave can be carried forward to the next year, up to a maximum of 5 days.\nSick leave is separate from annual leave, and employees get 10 paid sick days per year.\nA medical certificate is required for sick leave longer than 2 consecutive days.'), Document(metadata={'source': 'data\\hr_policy.txt'}, page_content='2. WORK FROM HOME POLICY\nEmployees may work from home up to 2 days per week, subject to manager approval.\nFully remote work arrangements require written approval from the department head.\nEmployees working from home must be reachable during core hours: 10 AM

In [25]:
print(chunks[1].page_content)

1. LEAVE POLICY
All full-time employees are entitled to 20 days of paid annual leave per calendar year.
Leave requests must be submitted through the HR portal at least 5 working days in advance.
Unused annual leave can be carried forward to the next year, up to a maximum of 5 days.
Sick leave is separate from annual leave, and employees get 10 paid sick days per year.
A medical certificate is required for sick leave longer than 2 consecutive days.


In [27]:
embeddings_model = JinaEmbeddings(model_name="jina-embeddings-v2-base-en")

print("EMB MODEL IS READY THE NAME IS ", embeddings_model.model_name)

EMB MODEL IS READY THE NAME IS  jina-embeddings-v2-base-en


In [29]:
vector_store = FAISS.from_documents(chunks, embeddings_model)


print("CHUNKS ARE STORED", vector_store.index.ntotal)

CHUNKS ARE STORED 9


In [31]:
test_query = "How many sick leavers employees get"


top_matches = vector_store.similarity_search(test_query, k=2)

print(f"Query : {test_query}\n")
for i, match in enumerate(top_matches, start=1):
    print(f"---- Match {i} ----")
    print(match.page_content)
    print()

Query : How many sick leavers employees get

---- Match 1 ----
1. LEAVE POLICY
All full-time employees are entitled to 20 days of paid annual leave per calendar year.
Leave requests must be submitted through the HR portal at least 5 working days in advance.
Unused annual leave can be carried forward to the next year, up to a maximum of 5 days.
Sick leave is separate from annual leave, and employees get 10 paid sick days per year.
A medical certificate is required for sick leave longer than 2 consecutive days.

---- Match 2 ----
7. HOLIDAYS
The company observes 12 public holidays every year, as per the official holiday calendar
published by HR at the start of each year.
Employees working on a public holiday are eligible for compensatory leave.



In [58]:
llm = ChatGroq(
    model="openai/gpt-oss-120b",
    temperature=0
)

In [59]:
llm.invoke('hi')

AIMessage(content='Hello! How can I help you today?', additional_kwargs={'reasoning_content': 'The user says "hi". We need to respond appropriately. No special instructions. Just greet.'}, response_metadata={'token_usage': {'completion_tokens': 38, 'prompt_tokens': 72, 'total_tokens': 110, 'completion_time': 0.079978458, 'completion_tokens_details': {'reasoning_tokens': 20}, 'prompt_time': 0.002753031, 'prompt_tokens_details': None, 'queue_time': 0.017882262, 'total_time': 0.082731489}, 'model_name': 'openai/gpt-oss-120b', 'system_fingerprint': 'fp_854fa9be4c', 'service_tier': 'on_demand', 'finish_reason': 'stop', 'logprobs': None, 'model_provider': 'groq'}, id='lc_run--019fc368-9871-7942-969f-ab7b28715561-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 72, 'output_tokens': 38, 'total_tokens': 110, 'output_token_details': {'reasoning': 20}})

In [60]:
retriever = vector_store.as_retriever(search_kwargs = {"k": 3})
def search_hr_policy(question: str) ->str:
    """
    Search the HR policy document for information about leave, work from home,
    probation, notice period, reimbursement, code of conduct, holidays, or exit process.
    
    """
    
    matching_chunks = retriever.invoke(question)
    return "\n\n".join(chunk.page_content for chunk in matching_chunks)

In [61]:
from langchain.agents import create_agent

hr_assistant = create_agent(
    model=llm,
    tools=[search_hr_policy],
    system_prompt= """ 
    You are a friendly HR assistant working for Acme Crop. 
    Always use the search_hr_policy tool to look up 
    facts before answering. 
    If the answer isn't in the search results, say you don't know "
    instead of guessing."
    """ 
)

print("HR assistant agent is ready to answer questions")

HR assistant agent is ready to answer questions


In [62]:
def ask_hr_assistante(question: str) -> str:
    print("="*60)
    print('QUESTION:', question)
    print("="*60)
    
    response = hr_assistant.invoke({"messages": [{"role": "user", "content": question}]})
    
    answer = response["messages"][-1].content
    print("ANSWER: ", answer)
    print("="*60)

In [69]:
ask_hr_assistante("tell me which organization you work for")

QUESTION: tell me which organization you work for
ANSWER:  I’m the friendly HR assistant here at **Acme Crop**. How can I help you today?


In [63]:
response = hr_assistant.invoke(
    {
        "messages": [
            {
             "role" :"user",
                "content": "tell me about leave policies how to apply for leave"
            }
            ]
    }
)

In [67]:
print(response["messages"][-2].content)

1. LEAVE POLICY
All full-time employees are entitled to 20 days of paid annual leave per calendar year.
Leave requests must be submitted through the HR portal at least 5 working days in advance.
Unused annual leave can be carried forward to the next year, up to a maximum of 5 days.
Sick leave is separate from annual leave, and employees get 10 paid sick days per year.
A medical certificate is required for sick leave longer than 2 consecutive days.

8. EXIT POLICY
Upon resignation or termination, employees must complete a clearance process involving
IT, Finance, and HR departments before their last working day.
Full and final settlement, including any pending reimbursements and leave encashment,
is processed within 45 days of the last working day.

7. HOLIDAYS
The company observes 12 public holidays every year, as per the official holiday calendar
published by HR at the start of each year.
Employees working on a public holiday are eligible for compensatory leave.


In [68]:
response

{'messages': [HumanMessage(content='tell me about leave policies how to apply for leave', additional_kwargs={}, response_metadata={}, id='7cf32c96-6e75-4369-989b-61c4767d65f5'),
  AIMessage(content='', additional_kwargs={'reasoning_content': 'We need to use search_hr_policy tool to look up info about leave policies and how to apply. Then answer based on results.', 'tool_calls': [{'id': 'fc_b16eea4f-76b5-43dc-947f-0bb17515d025', 'function': {'arguments': '{"question":"leave policies how to apply for leave"}', 'name': 'search_hr_policy'}, 'type': 'function'}]}, response_metadata={'token_usage': {'completion_tokens': 61, 'prompt_tokens': 211, 'total_tokens': 272, 'completion_time': 0.129923369, 'completion_tokens_details': {'reasoning_tokens': 27}, 'prompt_time': 0.048597845, 'prompt_tokens_details': None, 'queue_time': 0.193796269, 'total_time': 0.178521214}, 'model_name': 'openai/gpt-oss-120b', 'system_fingerprint': 'fp_3af2d41834', 'service_tier': 'on_demand', 'finish_reason': 'tool_ca